# Prompt 11: Libraries & Versioning

1. Build a minimal in-memory prompt registry
2. Load prompts by id@version
3. Default + canary + rollback
4. Per-tenant override
5. Observability: log prompt_id@version on every call
6. Exercises: write a YAML-driven registry for one feature

Deps: `pip install jinja2 anthropic openai pyyaml`

## 1. Minimal Registry

In [ ]:
from dataclasses import dataclass, field
from jinja2 import Environment, BaseLoader
import os, time, uuid

env = Environment(loader=BaseLoader(), trim_blocks=True, lstrip_blocks=True)

@dataclass
class PromptVersion:
    id: str
    version: str
    status: str        # draft | canary | stable | deprecated | retired
    template: str
    model: str
    baseline_accuracy: float | None = None

@dataclass
class RolloutRule:
    version: str
    tenant: str | None = None
    traffic_pct: int = 0   # 0..100

class PromptRegistry:
    def __init__(self):
        self.prompts: dict[str, dict[str, PromptVersion]] = {}
        self.rollout: dict[str, dict] = {}  # id -> {'default': str, 'rules': [RolloutRule]}

    def register(self, pv: PromptVersion):
        self.prompts.setdefault(pv.id, {})[pv.version] = pv

    def set_default(self, id, version):
        self.rollout.setdefault(id, {'default': None, 'rules': []})['default'] = version

    def add_rule(self, id, rule: RolloutRule):
        self.rollout.setdefault(id, {'default': None, 'rules': []})['rules'].append(rule)

    def resolve(self, id, tenant: str | None = None, _rng=None) -> PromptVersion:
        cfg = self.rollout.get(id, {'default': None, 'rules': []})
        # tenant rule wins
        for rule in cfg['rules']:
            if rule.tenant and rule.tenant == tenant:
                return self.prompts[id][rule.version]
        # traffic percentage
        import random
        rng = _rng or random.random()
        cum = 0
        for rule in cfg['rules']:
            if rule.traffic_pct > 0:
                cum += rule.traffic_pct / 100.0
                if rng < cum:
                    return self.prompts[id][rule.version]
        # default
        return self.prompts[id][cfg['default']]

registry = PromptRegistry()

## 2. Register a Few Versions

In [ ]:
registry.register(PromptVersion(
    id='invoice_extract',
    version='v1',
    status='retired',
    template='Extract JSON: {"vendor":str,"total":num}\n\n{{ text }}',
    model='gpt-4o-mini',
    baseline_accuracy=0.75,
))

registry.register(PromptVersion(
    id='invoice_extract',
    version='v2',
    status='stable',
    template='''Extract JSON matching:
{"vendor":string,"total_usd":number,"due":"YYYY-MM-DD" or null}

Input: {{ text }}
Return JSON only.''',
    model='claude-sonnet-4-6',
    baseline_accuracy=0.92,
))

registry.register(PromptVersion(
    id='invoice_extract',
    version='v3',
    status='canary',
    template='''Extract JSON:
{"vendor":string,"total_usd":number,"due":"YYYY-MM-DD" or null,"due_source":"explicit|not_found"}

Do not infer dates not in the text. Input: {{ text }}''',
    model='claude-sonnet-4-6',
    baseline_accuracy=0.94,
))

registry.set_default('invoice_extract', 'v2')
registry.add_rule('invoice_extract', RolloutRule(version='v3', traffic_pct=10))
registry.add_rule('invoice_extract', RolloutRule(version='v3', tenant='beta-tester-inc'))

## 3. Resolve + Render

In [ ]:
counts = {'v2': 0, 'v3': 0}
import random
for i in range(200):
    v = registry.resolve('invoice_extract', _rng=random.random()).version
    counts[v] += 1
print('default + 10% canary (v3) across 200 calls:')
print(counts)

beta = registry.resolve('invoice_extract', tenant='beta-tester-inc')
print(f'\nbeta tenant always gets: {beta.version}')

## 4. End-to-End Call with Logging

In [ ]:
def call_llm(pv: PromptVersion, rendered: str) -> dict:
    t0 = time.perf_counter()
    if os.getenv('ANTHROPIC_API_KEY') and 'claude' in pv.model:
        from anthropic import Anthropic
        r = Anthropic().messages.create(model=pv.model, max_tokens=300, temperature=0, messages=[{'role':'user','content':rendered}])
        text = r.content[0].text; in_tok=r.usage.input_tokens; out_tok=r.usage.output_tokens
    elif os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=pv.model.replace('claude-sonnet-4-6','gpt-4o-mini'),
            max_tokens=300, temperature=0,
            messages=[{'role':'user','content':rendered}])
        text = r.choices[0].message.content; in_tok=r.usage.prompt_tokens; out_tok=r.usage.completion_tokens
    else:
        text='{}'; in_tok=0; out_tok=0
    return dict(text=text, in_tok=in_tok, out_tok=out_tok, ms=(time.perf_counter()-t0)*1000)

def run(id, variables, tenant=None):
    pv = registry.resolve(id, tenant=tenant)
    template = env.from_string(pv.template)
    rendered = template.render(**variables)
    result = call_llm(pv, rendered)
    # Log every call with prompt_id@version, tenant, cost, latency
    log_record = {
        'request_id': str(uuid.uuid4())[:8],
        'prompt_id': id,
        'prompt_version': pv.version,
        'model': pv.model,
        'tenant': tenant,
        'in_tok': result['in_tok'],
        'out_tok': result['out_tok'],
        'latency_ms': round(result['ms']),
    }
    print(log_record)
    return result

for tenant in [None, 'beta-tester-inc']:
    print(f'\n--- tenant={tenant} ---')
    run('invoice_extract', {'text':'Bill from NovaCore. Total: $438.00. Due: 2026-05-10.'}, tenant=tenant)

## 5. Rollback Workflow

Imagine v3 regresses (caught by monitoring). Rollback is a single config change.

In [ ]:
# Before rollback
print('before: default =', registry.rollout['invoice_extract']['default'])

# Remove the canary rule (simplest rollback)
registry.rollout['invoice_extract']['rules'] = [r for r in registry.rollout['invoice_extract']['rules'] if r.traffic_pct == 0]

counts = {'v2':0,'v3':0}
for _ in range(100):
    v = registry.resolve('invoice_extract', _rng=random.random()).version
    counts[v] += 1
print('after removing 10% canary:', counts)

# Beta tenants still get v3 until we also remove the tenant rule
print('beta still on:', registry.resolve('invoice_extract', tenant='beta-tester-inc').version)

## 6. Exercise: YAML-Driven Registry

For one feature:

1. Put `prompts/<feature>/v1.jinja`, `v2.jinja`, etc.
2. Write `prompts/<feature>/metadata.yaml` with owner, versions, rollout.
3. Write `registry_loader.py` that reads those files and populates `PromptRegistry`.
4. Build a CLI: `python -m promptctl deploy <feature> --version v3 --canary 10`.
5. Log `prompt_id@version` on every call; dashboard cuts by version.

You've just built the core of a production prompt-management system.

## 7. Takeaways

- **Registry > inline strings.**
- **Metadata + tests** travel with every prompt.
- **Traffic rules** decouple prompt deploys from code deploys.
- **Log version on every call** for observability and rollback.
- **Retire, don't delete.**

Next: **12 — Multi-Modal Prompting** — the final session of this module.